<b>To obtain the maximum points via the autograder, your submitted model must outperform the following metrics:</b>

<u>Test Accuracy:</u> 0.983

<u>Production Accuracy:</u> 0.983

In [1]:
import pandas as pd
from joblib import dump
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV

from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
pitches_train = pd.read_parquet('https://lab.cs307.org/pitches/data/pitches-train.parquet')
pitches_test = pd.read_parquet('https://lab.cs307.org/pitches/data/pitches-test.parquet')

In [3]:
pitches_train

,pitch_type,release_speed,release_spin_rate,pfx_x,pfx_z,stand
0,FF,97.3,2432.0,-0.80,1.61,L
1,FF,97.9,2385.0,-0.93,1.22,L
2,FF,96.6,2390.0,-0.97,1.38,L
3,FS,86.4,1492.0,-1.27,0.50,L
4,FS,88.9,1778.0,-1.14,0.68,R
...,...,...,...,...,...,...
2863,FS,87.9,1849.0,-1.60,0.09,R
2864,FS,85.5,1876.0,-1.64,0.18,R
2865,FF,94.0,2474.0,-1.01,1.73,R
2866,FF,93.9,2429.0,-0.96,1.75,R


In [4]:
pitches_test

,pitch_type,release_speed,release_spin_rate,pfx_x,pfx_z,stand
0,FF,96.0,2387,-1.00,1.45,L
1,FS,84.6,1736,-0.63,-0.11,R
2,FS,85.9,1732,-1.31,0.55,R
3,FF,95.4,2304,-1.01,1.22,R
4,FF,96.7,2307,-1.11,1.17,R
...,...,...,...,...,...,...
1801,FF,95.9,2345,-0.92,1.61,L
1802,SL,87.1,2459,0.23,0.42,L
1803,FF,96.0,2341,NaN,1.47,L
1804,FF,95.0,2348,-1.12,1.53,L


In [ ]:
X_train = pitches_train.drop('pitch_type', axis = 1)
y_train = pitches_train['pitch_type']
X_test = pitches_test.drop('pitch_type', axis = 1)
y_test = pitches_test['pitch_type']

In [ ]:
#data shape
n_samples = X_train.shape[0]
n_features = X_train.shape[1]
print('number of samples: ' + str(n_samples) + '\nnumber of features: ' + str(n_features))

In [ ]:
#target balance
len_pitch_type = len(pitches_train)

ff_count = len(pitches_train[pitches_train.pitch_type == 'FF'])
fs_count = len(pitches_train[pitches_train.pitch_type == 'FS'])
si_count = len(pitches_train[pitches_train.pitch_type == 'SI'])
sl_count = len(pitches_train[pitches_train.pitch_type == 'SL'])

print('Pitch Type counts:' + 'ff_count = ' + str(ff_count) +
      '\nfs_count = ' + str(fs_count) + 
      '\nsi_count = ' + str(si_count) + 
      '\nsl_count = ' + str(sl_count) + 
      '\n_____________')

ff_prop = ff_count / len_pitch_type
fs_prop = fs_count / len_pitch_type
si_prop = si_count / len_pitch_type
sl_prop = sl_count / len_pitch_type

print('ff_prop = ' + str(ff_prop) +
      '\nfs_prop = ' + str(fs_prop) + 
      '\nsi_prop = ' + str(si_prop) + 
      '\nsl_prop = ' + str(sl_prop))


In [ ]:
#velocity by pitch type
pitches_ff = pitches_train[pitches_train.pitch_type == 'FF']
mv_ff = pitches_ff.release_speed.mean()
std_ff = pitches_ff.release_speed.std()

pitches_fs = pitches_train[pitches_train.pitch_type == 'FS']
mv_fs = pitches_fs.release_speed.mean()
std_fs = pitches_fs.release_speed.std()

pitches_si = pitches_train[pitches_train.pitch_type == 'SI']
mv_si = pitches_si.release_speed.mean()
std_si = pitches_si.release_speed.std()

pitches_sl = pitches_train[pitches_train.pitch_type == 'SL']
mv_sl = pitches_sl.release_speed.mean()
std_sl = pitches_sl.release_speed.std()

print('Means (FF, FS, SI, SL): ' + 
      str(mv_ff) + ', ' + 
      str(mv_fs) + ', ' + 
      str(mv_si) + ', ' + 
      str(mv_sl))

print('STD (FF, FS, SI, SL): ' +
      str(std_ff) + ', ' + 
      str(std_fs) + ', ' + 
      str(std_si) + ', ' + 
      str(std_sl))

In [ ]:
#spin rate by pitch type
ms_ff = pitches_ff.release_spin_rate.mean()
ms_fs = pitches_fs.release_spin_rate.mean()
ms_si = pitches_si.release_spin_rate.mean()
ms_sl = pitches_sl.release_spin_rate.mean()

rstd_ff = pitches_ff.release_spin_rate.std()
rstd_fs = pitches_fs.release_spin_rate.std()
rstd_si = pitches_si.release_spin_rate.std()
rstd_sl = pitches_sl.release_spin_rate.std()

print('Spin Rate\n'+'Means (FF, FS, SI, SL): ' + 
      str(ms_ff) + ', ' + 
      str(ms_fs) + ', ' + 
      str(ms_si) + ', ' + 
      str(ms_sl))

print('STD (FF, FS, SI, SL): ' +
      str(rstd_ff) + ', ' + 
      str(rstd_fs) + ', ' + 
      str(rstd_si) + ', ' + 
      str(rstd_sl))


# <u>Model:</u>
You are tasked with developing a model to assist with automatically displaying the pitch type for each pitch in real-time, both in the stadium, and on the television broadcast. You have access to data on every pitch thrown in MLB to date, including characteristics of the pitch such as its velocity and rotation, as well as the type of pitch thrown. Additionally, tracking technology in each stadium will provide data on (at least) the speed and rotation of each pitch, in real-time. You are asked to create a proof-of-concept for a single pitcher, Kevin Gausman, but your process should be designed such that it can easily be applied to other pitchers as well.


- model must have a fit method
- model must have a predict method
    - if classification is the task, it must have a predict_proba method



### <u>Model Submission:</u>
from joblib import dump

dump(model_object, "filename.joblib")

In [ ]:
X_train

In [ ]:
#lucy 
numeric_transformer = make_pipeline(SimpleImputer(strategy = "median"),StandardScaler())
categorical_transformer = make_pipeline(SimpleImputer(strategy = "most_frequent"),OneHotEncoder())

numeric_features = ['release_speed', 'release_spin_rate', 'pfx_x', 'pfx_z']
categorical_features = ['stand']

preprocessor = make_column_transformer((numeric_transformer, numeric_features),(categorical_transformer, categorical_features))

preprocessor.fit(X_train)
preprocessor.transform(X_train)

pitches = make_pipeline(preprocessor,KNeighborsClassifier(n_neighbors = 100))
pitches.fit(X_train, y_train)
y_pred = pitches.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(str(accuracy))

In [ ]:
#dump(pitches, 'pitches.joblib')

In [ ]:
#lyons

numeric_transformer = make_pipeline(SimpleImputer(strategy = "mean"), StandardScaler())
categorical_transformer = make_pipeline(SimpleImputer(strategy = "most_frequent"), OneHotEncoder())

numeric_features = X_train.select_dtypes(include = ['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include = ['object', 'category']).columns

preprocessor = make_column_transformer((numeric_transformer, numeric_features),
                                       (categorical_transformer, categorical_features))
preprocessor.fit(X_train)
preprocessor.transform(X_train)

model = make_pipeline(preprocessor, KNeighborsClassifier(n_neighbors = 300))
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(str(accuracy))

dump(model, 'pitches.joblib')
